In [ ]:
# Biblioteca para leitura e manipulação dos dados.
import pandas as pd
# Biblioteca para montar o caminho dos arquivos.
import os

# Pasta onde estão os 7 arquivos CSV baixados.
pasta = "bases_20_26"
# Lista dos anos que serão concatenados.
anos = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
# Nome do arquivo final, já consolidado.
saida = "BPS_20_26_LuizFernandoDeJesusSilvaHomem.csv"

In [2]:
# Dicionário de mapeamento: nome da coluna em 2020 -> nome padrão usado em 2021-2026.
# Necessário porque o arquivo de 2020 usa uma nomenclatura diferente dos demais anos.
mapeamento_2020 = {
    "sg_uf": "uf",
    "ds_esfera": "esfera",
    "dt_compra": "compra",
    "dt_insercao": "insercao",
    "co_catmat": "codigo_br",
    "ds_item": "descricao_catmat",
    "fg_generico": "generico",
    "tp_compra": "tipo_compra",
    "sg_unidade_medida": "unidade_medida",
    "no_fornecedor": "fornecedor",
    "no_fabricante": "fabricante",
    "qt_medicamento": "qtd_itens_comprados",
    "no_instituicao": "nome_instituicao",
    "no_municipio": "municipio_instituicao",
    "un_medida_capacidade": "unidade_fornecimento_capacidade",
    "un_fornecimento": "unidade_fornecimento",
    "registro_anvisa": "anvisa",
    "modalidade": "modalidade_compra",
    "vl_capacidade": "capacidade",
    "vl_preco_unitario": "preco_unitario",
    "vl_preco_total": "preco_total",
}

In [3]:
# Forçando as colunas de CNPJ a serem lidas como texto (str), não como número.
dtype_cnpj = {
    "cnpj_instituicao": str,
    "cnpj_fornecedor": str,
    "cnpj_fabricante": str,
}

# Lista vazia que vai guardar o DataFrame de cada ano, antes de juntar tudo.
dataframes = []

In [4]:
# Passando por cada ano, um de cada vez.
for ano in anos:
    # Montando o caminho completo do arquivo daquele ano.
    caminho = os.path.join(pasta, f"{ano}.csv")

    # Tentando ler o arquivo completo com separador ";", já aplicando o dtype dos CNPJs.
    try:
        df = pd.read_csv(caminho, sep=";", encoding="utf-8", low_memory=False, dtype=dtype_cnpj)
    # Se não der certo com ";", tenta com "," no lugar.
    except Exception:
        df = pd.read_csv(caminho, sep=",", encoding="utf-8", low_memory=False, dtype=dtype_cnpj)

    # Só o ano de 2020 precisa ser renomeado (os outros anos já usam o nome padrão).
    if ano == 2020:
        df = df.rename(columns=mapeamento_2020)

    # Guardando o DataFrame desse ano na lista.
    dataframes.append(df)
    # Mostrando quantas linhas e colunas foram lidas nesse ano.
    print(f"{ano}: {len(df)} linhas, {len(df.columns)} colunas lidas")

2020: 84919 linhas, 36 colunas lidas
2021: 83622 linhas, 25 colunas lidas
2022: 88991 linhas, 25 colunas lidas
2023: 31992 linhas, 25 colunas lidas
2024: 26258 linhas, 25 colunas lidas
2025: 26215 linhas, 25 colunas lidas
2026: 819 linhas, 25 colunas lidas


In [5]:
# Juntando os 7 DataFrames em um só.
# ignore_index=True refaz a numeração das linhas do zero (0, 1, 2, ...).
# sort=False mantém a ordem das colunas como estão, sem reordenar por nome.
# Colunas que não existem em algum ano (ex: as exclusivas de 2020) viram NaN nesse ano.
base_final = pd.concat(dataframes, ignore_index=True, sort=False)

# Conferindo o tamanho final da base.
print(f"\nBase final: {len(base_final)} linhas, {len(base_final.columns)} colunas")
print("\nColunas finais:")
print(list(base_final.columns))


Base final: 342816 linhas, 36 colunas

Colunas finais:
['ano_compra', 'cnpj_instituicao', 'uf', 'esfera', 'compra', 'insercao', 'validade_compra', 'codigo_br', 'descricao_catmat', 'co_pdm', 'co_grupo', 'no_grupo', 'co_classe', 'no_classe', 'generico', 'tipo_compra', 'unidade_medida', 'cnpj_fornecedor', 'fornecedor', 'cnpj_fabricante', 'fabricante', 'qtd_itens_comprados', 'ds_observacao', 'nome_instituicao', 'municipio_instituicao', 'unidade_fornecimento_capacidade', 'no_pdm', 'nu_processo_compra', 'nu_ata', 'unidade_fornecimento', 'anvisa', 'modalidade_compra', 'capacidade', 'preco_unitario', 'preco_total', 'co_seq_bps']


In [6]:
# Conferindo se os CNPJs realmente vieram como texto (esperado: "object", não "int64").
print("\nConferindo tipo das colunas de CNPJ (devem ser 'object', ou seja, texto):")
print(base_final[["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]].dtypes)


Conferindo tipo das colunas de CNPJ (devem ser 'object', ou seja, texto):
cnpj_instituicao    str
cnpj_fornecedor     str
cnpj_fabricante     str
dtype: object


In [7]:
# Salvando a base concatenada em CSV, com ";" como separador (padrão que usamos em todo o projeto).
base_final.to_csv(saida, index=False, sep=";", encoding="utf-8")
print(f"\nArquivo salvo como: {saida}")


Arquivo salvo como: BPS_20_26_LuizFernandoDeJesusSilvaHomem.csv
